In [3]:
 
# CELL 1: Imports and Paths
 

import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('./data/raw')
PATIENT_DIR = Path("./data/raw/AngioCAD_Dataset")
OUTPUT_DIR = Path('./data/processed')
FRAMES_DIR = OUTPUT_DIR / 'frames'
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
MAX_FRAMES_PER_SERIES = 10

print(f"Frames will be saved to: {FRAMES_DIR}")

Frames will be saved to: data\processed\frames


In [4]:
 
# CELL 2: Frame Loading Utility
 

def load_series_frames(series_path, max_frames=MAX_FRAMES_PER_SERIES):
    """Load evenly-sampled frames from a series folder."""
    exts = ('*.jpg', '*.jpeg', '*.png')
    files = []
    for e in exts:
        files.extend(sorted(series_path.glob(e)))
    
    if len(files) == 0:
        return []
    
    # Evenly sample frames
    if len(files) <= max_frames:
        chosen = files
    else:
        idx = np.linspace(0, len(files) - 1, max_frames).astype(int)
        chosen = [files[i] for i in idx]
    
    frames = []
    for f in chosen:
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, IMG_SIZE)
        frames.append(img)
    return frames

def split_series(s):
    if pd.isna(s):
        return []
    return [x.strip() for x in str(s).split('-') if x.strip()]

In [5]:
  
# CELL 3: Load Merged Data
  

df = pd.read_csv(OUTPUT_DIR / 'clinical_features_merged.csv')

# Identify right/left column names (after merge they get suffixed)
right_col = 'Right Coronary Series_y' if 'Right Coronary Series_y' in df.columns else 'Right Coronary Series'
left_col  = 'Left Coronary Series_y'  if 'Left Coronary Series_y'  in df.columns else 'Left Coronary Series'

print(f"Using columns: right='{right_col}', left='{left_col}'")
print(f"Total patients: {len(df)}")

Using columns: right='Right Coronary Series_y', left='Left Coronary Series_y'
Total patients: 377


In [6]:
   
# CELL 4: Extract Frames From All Patients
   

records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Extracting frames'):
    pid = str(int(row['ID']))
    patient_path = PATIENT_DIR / pid
    
    if not patient_path.exists():
        continue
    
    right_series = split_series(row.get(right_col))
    left_series  = split_series(row.get(left_col))
    
    # ---- Right Coronary ----
    for s in right_series:
        sp = patient_path / s
        if not sp.exists():
            continue
        frames = load_series_frames(sp)
        for i, fr in enumerate(frames):
            out_name = f"{pid}_R_{s}_{i}.png"
            out_path = FRAMES_DIR / out_name
            cv2.imwrite(str(out_path), fr)
            records.append({
                'ID': pid, 'side': 'R', 'series': s,
                'frame_idx': i, 'path': str(out_path),
            })
    
    # ---- Left Coronary ----
    for s in left_series:
        sp = patient_path / s
        if not sp.exists():
            continue
        frames = load_series_frames(sp)
        for i, fr in enumerate(frames):
            out_name = f"{pid}_L_{s}_{i}.png"
            out_path = FRAMES_DIR / out_name
            cv2.imwrite(str(out_path), fr)
            records.append({
                'ID': pid, 'side': 'L', 'series': s,
                'frame_idx': i, 'path': str(out_path),
            })

print(f"\nExtracted {len(records)} frames")

Extracting frames: 100%|██████████| 377/377 [02:14<00:00,  2.81it/s]


Extracted 13353 frames


In [7]:
    
# CELL 5: Save Frame Index
    

frames_df = pd.DataFrame(records)
frames_df.to_csv(OUTPUT_DIR / 'frames_index.csv', index=False)

print(f"Unique patients: {frames_df['ID'].nunique()}")
print(f"Total frames   : {len(frames_df)}")
print(f"\nFrames per side:")
print(frames_df['side'].value_counts())

frames_df.head()

Unique patients: 376
Total frames   : 13353

Frames per side:
side
L    7137
R    6216
Name: count, dtype: int64


,ID,side,series,frame_idx,path
0,1,R,1,0,data\processed\frames\1_R_1_0.png
1,1,R,1,1,data\processed\frames\1_R_1_1.png
2,1,R,1,2,data\processed\frames\1_R_1_2.png
3,1,R,1,3,data\processed\frames\1_R_1_3.png
4,1,R,1,4,data\processed\frames\1_R_1_4.png
